In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
df = pd.read_csv('15-gym_crowdedness.csv')

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.shape

In [ ]:
df.describe()

In [ ]:
df.isnull().sum()

In [ ]:
# we can convert the 'date' column. from object to datetime.

In [ ]:
df['date'] = pd.to_datetime(df['date'], utc=True)

In [ ]:
df.info()

In [ ]:
# we can create a 'year' column from datetime. because we have already month, day and hour columns.

In [ ]:
df['year'] = df['date'].dt.year

In [ ]:
df.drop('date', axis=1, inplace=True)

In [ ]:
df.head()

In [ ]:
df['year'].unique()

In [ ]:
sns.lineplot(x=df['hour'], y=df['number_people'])
plt.show()

In [ ]:
sns.boxplot(y=df['number_people'])
plt.show()

In [ ]:
sns.barplot(x=df['day_of_week'], y=df['number_people'])
plt.show()

In [ ]:
sns.barplot(x=df['is_start_of_semester'], y=df['number_people'])
plt.show()

In [ ]:
sns.barplot(x=df['is_holiday'], y=df['number_people'])
plt.show()

In [ ]:
sns.lineplot(x=df['temperature'], y=df['number_people'])
plt.show()

In [ ]:
sns.lineplot(x=df['month'], y=df['number_people'])
plt.show()

In [ ]:
sns.regplot(x=df['hour'], y=df['number_people'])

In [ ]:
sns.heatmap(df.corr(numeric_only=True), annot=True, cmap="coolwarm")
plt.show()

In [ ]:
# in accordance with this heatmap result can drop the 'timestamp' column.

In [ ]:
df.drop("timestamp", axis=1, inplace=True)

In [ ]:
df.head()

In [ ]:
# test scores are bad so we'll try a solution
df['lag1'] = df['number_people'].shift(1)
df['lag24'] = df['number_people'].shift(24)
df['lag168'] = df['number_people'].shift(168)

df = df.dropna().reset_index(drop=True)

In [ ]:
X = df.drop("number_people", axis=1) 
y = df['number_people']

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.15, random_state=15, shuffle=False)

In [ ]:
from sklearn.preprocessing import StandardScaler

In [ ]:
scaler = StandardScaler()

In [ ]:
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [ ]:
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [ ]:
def calculate_metrics(true, predicted):
    mae = mean_absolute_error(true, predicted)
    mse = mean_squared_error(true, predicted)
    rmse = np.sqrt(mean_squared_error(true, predicted))
    r2_square = r2_score(true, predicted)
    return mae, rmse, r2_square

In [ ]:
models = {
    "Linear Regression" : LinearRegression(),
    "Ridge" : Ridge(),   
    "Lasso" : Lasso(),
    "K-Neighbors Regressor" : KNeighborsRegressor(),
    "Decision Tree Regressor" : DecisionTreeRegressor(),
    "Random Forest Regressor" : RandomForestRegressor()
}

In [ ]:
for i in range(len(list(models))):
    model = list(models.values())[i]
    model.fit(X_train, y_train)

    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)

    model_train_mae, model_train_rmse, model_train_r2 = calculate_metrics(y_train, y_train_pred)
    model_test_mae, model_test_rmse, model_test_r2 = calculate_metrics(y_test, y_test_pred)

    print(list(models.values())[i])
    
    print("Evaluation for Training Set")
    print("Mean Absolute Error :", model_train_mae)
    print("RMSE :", model_train_rmse)
    print("R2 Score :", model_train_r2)

    print("------------------------")

    print("Evaluation for Test Set")
    print("Mean Absolute Error :", model_test_mae)
    print("RMSE :", model_test_rmse)
    print("R2 Score :", model_test_r2)

    print("------------------------")
    print("\n")

In [ ]:
#Hyperparameter Tuning

In [ ]:
knn_params = {"n_neighbors" : [2, 3, 10, 20, 30]}

rf_params = {
    "n_estimators" : [10, 50, 150, 200, 300, 1000],
    "max_depth" : [None, 2, 3, 5, 7, 10, 20],
    "max_features" : ["sqrt", "log2", None],
    "min_samples_split" : [2, 3, 5, 10, 20]
}

In [ ]:
from sklearn.model_selection import RandomizedSearchCV

In [ ]:
random_cv_models = [
    ("KNN", KNeighborsRegressor(), knn_params),
    ("RF", RandomForestRegressor(), rf_params)
]

In [ ]:
for name, model, params in random_cv_models:
    randomcv = RandomizedSearchCV(estimator=model, param_distributions=params, n_jobs=-1, cv=5)
    randomcv.fit(X_train, y_train)
    print("best params for :", name, randomcv.best_params_)

In [ ]:
models = {
    "K-Neighbors Regressor" : KNeighborsRegressor(),
    "Random Forest Regressor" : RandomForestRegressor()
}

In [ ]:
for i in range(len(list(models))):
    model = list(models.values())[i]
    model.fit(X_train, y_train)

    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)

    model_train_mae, model_train_rmse, model_train_r2 = calculate_metrics(y_train, y_train_pred)
    model_test_mae, model_test_rmse, model_test_r2 = calculate_metrics(y_test, y_test_pred)

    print(list(models.values())[i])
    
    print("Evaluation for Training Set")
    print("Mean Absolute Error :", model_train_mae)
    print("RMSE :", model_train_rmse)
    print("R2 Score :", model_train_r2)

    print("------------------------")

    print("Evaluation for Test Set")
    print("Mean Absolute Error :", model_test_mae)
    print("RMSE :", model_test_rmse)
    print("R2 Score :", model_test_r2)

    print("------------------------")
    print("\n")